# Paired C/D evidence-selection pilot (v5)
Select T4 GPU and Run all. Three cases per condition. Qwen selects a class and evidence IDs and writes short prose; Python attaches exact numerical evidence from the selected case. Raw selections remain saved and derived evidence is identified as deterministic source lookup. This changes output construction equally in C and D; compare within this version. Explanation support remains separately reviewed.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
from google.colab import drive
ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Drive connection failed. Select Runtime > Disconnect and delete runtime, reconnect and rerun this cell. Complete Google authorisation. No diagnosis has started.') from exc
if not (ROOT / 'Files/phm_selection_pilot.py').is_file():
    raise FileNotFoundError(f'Drive is connected but the updated project is missing at {ROOT}. Finish sync or locate CP5403 in the Files panel. Do not force-remount.')
os.chdir(ROOT)
EXPECTED = {'phm_condition_d.py': '65923a51de641781736160e31108e668d5fa7fbed37938b1bb79722e28e447ed', 'phm_selection_pilot.py': '3eaef1b55c56a3b6a2977d564b54cc0ebdd0deab1548680072c2f569c29eeafb', 'phm_evidence_selection.py': 'bb8cd412496f3d055d11b6d0f4ac47e970f24d43b4c19493d9ff57fca8b4fac9', 'phm_literature_index.py': '725709141868ae494f1c515f67df5aec4451c0d137d461b3991c7ba436fb7f37', 'phm_diagnosis.py': 'a0b2af61d7d1a1c16512359f1e8029aae48380fc4b698c188f1f8caacf14b2fd', 'phm_diagnosis_qwen.py': 'f3c9938f7fd2638de842139db5f5d2716c57f4839860291279e45c4d47e049bb', 'test_phm_selection.py': '06b9ef19b122f7c9f745f12cb27a0c1c3fc3ffb0026c41adb57cd08ad8777c06', 'phm_output_schema.py': '01913f0c804300723c841d7c689be5105749c1e3d9c0bc110d7a9b204ea048e9'}
for name, expected in EXPECTED.items():
    path = ROOT / 'Files' / name
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == expected, f'File not fully synced: {name}'
INDEX = ROOT / 'Datasets/PHM15-literature-D-approved-v1'
manifest = json.loads((INDEX / 'index_manifest.json').read_text(encoding='utf-8'))
for name, key in [('approved_chunks.jsonl','chunks_sha256'), ('embeddings.npy','embeddings_sha256'), ('model_assets/minilm.onnx','onnx_sha256')]:
    assert hashlib.sha256((INDEX / name).read_bytes()).hexdigest() == manifest[key], f'Index not fully synced: {name}'
print('Project and approved index verified:', manifest['chunk_count'], 'chunks')


Mounted at /content/drive
Project and approved index verified: 24 chunks


In [2]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Files/requirements-literature-colab.txt', 'onnxruntime==1.31.1', 'lm-format-enforcer==0.11.3'], check=True)
print('Dependencies installed.')


GPU: Tesla T4
Dependencies installed.


In [3]:
for test in ['test_phm_diagnosis.py', 'test_phm_selection.py']:
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files', '-p', test, '-q'], check=True)
subprocess.run([sys.executable, 'Files/phm_selection_pilot.py', '--root', str(ROOT), '--max-cases', '3', '--prepare-only'], check=True)
print('Paired evidence and source checks passed. The final cell runs Qwen.')


Paired evidence and source checks passed. The final cell runs Qwen.


In [4]:
subprocess.run([sys.executable, 'Files/phm_selection_pilot.py', '--root', str(ROOT), '--max-cases', '3'], check=True)
latest = json.loads((ROOT / 'Outputs/LATEST_RUN.json').read_text(encoding='utf-8'))
print('Latest report:', latest)
print('Let Google Drive finish syncing, then ask for the paired C/D pilot review.')


Latest report: {'run_id': '20261121T175502_paired_CD_selection_pilot_15573281', 'run_directory': 'Outputs/runs/20261121T175502_paired_CD_selection_pilot_15573281', 'status': 'completed', 'updated_utc': '2026-11-21T18:03:46.517673+00:00'}
Let Google Drive finish syncing, then ask for the paired C/D pilot review.
